# Phase 01 — Data Audit and Output Contracts

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Understand source data quality and align model outputs with product/API contracts before feature or label work.

This notebook is the Phase 1 source of truth. It writes `reports/phase_01_data_audit_contracts.json` so later label, normalization, baseline, and training phases inherit the same input-field decisions and product boundary.

## Purpose
Document and verify Phase 01 — Data Audit and Output Contracts in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 01.data.audit.contracts notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 01 — Data Audit and Output Contracts.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Contract boundary

Model/training owns only stable, evidence-grounded core signals:

- `jobFitAlignment.score` and grounded alignment signals such as matched skills, missing skills, role match, experience match, and requirement coverage.
- `atsFriendliness.score` and detected ATS issues based on parseability, structure, contact/date detection, metric evidence, and formatting risk.
- `overallImpression` as a grounded summary signal derived from observed alignment and ATS evidence.
- Recommendation scoring/ranking signals only for backend-provided candidate jobs.

Backend/API wrapper owns request validation, auth, persistence, CV file ownership, OpenAI wrapper orchestration, `topActionables`, `sectionReviews`, generated CV availability, final job detail hydration, and public response shaping.


## Shared setup

### Purpose
Define deterministic paths, source-field policies, output-boundary tables, and reporting helpers used by every Phase 1 step.

### Required input
Repository root with `legacy/dataset/indotech_job_cleaned.csv`, `legacy/dataset/techtalent_profile_cleaned.csv`, `legacy/artifacts/pairs.parquet`, `references/docs/generated/openapi.json`, `references/docs/modules/ai-cv-analyzer.md`, `references/docs/integrations/model-api.md`, and the Phase 0 report.

### Action
Load standard-library helpers only. No model training, label generation, feature extraction, or artifact mutation occurs in this notebook.

### Expected output
Reusable helpers and static policy tables for schema review, identifier checks, contract mapping, boundary separation, and readiness decisions.

### Verification
The setup cell must run without optional notebook dependencies. Generated output must be limited to `reports/phase_01_data_audit_contracts.json`.


In [7]:
from __future__ import annotations

import csv
import json
import math
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TODOS.md").exists() and (candidate / "legacy").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd().resolve())
LEGACY = ROOT / "legacy"
REPORTS = ROOT / "reports"
JOBS_CSV = LEGACY / "dataset" / "indotech_job_cleaned.csv"
PROFILES_CSV = LEGACY / "dataset" / "techtalent_profile_cleaned.csv"
PAIRS_PARQUET = LEGACY / "artifacts" / "pairs.parquet"
OPENAPI_JSON = ROOT / "references" / "docs" / "generated" / "openapi.json"
PHASE0_REPORT = REPORTS / "phase_00_reproducibility_snapshot.json"
PHASE1_REPORT = REPORTS / "phase_01_data_audit_contracts.json"

JOB_FIELD_POLICIES = {
    "job_id": ("Stable job identifier used for joins, split diagnostics, and backend candidate binding.", "identifier_only", "Never use as model feature; use for grouping, deduplication, and output validation."),
    "company_name": ("Company display name used by backend hydration and audit examples.", "use_with_caution", "Can introduce employer memorization; avoid as model feature unless explicit bias review exists."),
    "title": ("Raw job title text used for role-alignment evidence.", "feature", "Safe after normalization and leakage review."),
    "normalized_title": ("Canonical role title used for role-family grouping and slice analysis.", "feature", "Safe after title normalization policy is versioned."),
    "category": ("Job category used for role-family diagnostics and pair balancing.", "feature", "Safe for slice analysis; use as model feature only after category taxonomy is fixed."),
    "description": ("Job description text used for semantic alignment and requirement coverage.", "feature", "Safe after PII and stale-content checks."),
    "requirement_summary": ("Condensed requirement text used for requirement coverage and missing-signal extraction.", "feature", "Safe if derived from job record, not from labels or outcomes."),
    "work_type": ("Work arrangement signal used by backend/preference matching, not core job-fit training unless approved.", "wrapper_or_backend", "Backend preference feature; keep outside core scorer for current roadmap."),
    "employment_type": ("Employment contract type used for filtering and optional preference diagnostics.", "wrapper_or_backend", "Backend-owned filter unless explicit training requirement is added."),
    "experience_level": ("Job seniority level used for experience-match labels and normalization checks.", "feature", "Safe only after Phase 3 experience mapping covers every value."),
    "province": ("Job province used for backend location preference and data-slice diagnostics.", "wrapper_or_backend", "Backend preference/hydration field; avoid in core model until location policy is approved."),
    "city": ("Job city used for backend location preference and data-slice diagnostics.", "wrapper_or_backend", "Backend preference/hydration field; avoid in core model until location policy is approved."),
    "salary_min": ("Minimum salary metadata used for backend display/filtering and optional preference checks.", "wrapper_or_backend", "Do not use in model before salary fairness and missingness policy exists."),
    "salary_max": ("Maximum salary metadata used for backend display/filtering and optional preference checks.", "wrapper_or_backend", "Do not use in model before salary fairness and missingness policy exists."),
    "salary_currency": ("Salary currency metadata used for backend display/filtering.", "wrapper_or_backend", "Backend-owned display/filter field."),
    "salary_display": ("Human salary display text used for backend hydration only.", "wrapper_or_backend", "Do not train on display strings."),
    "skills_top_10_names": ("Raw top skills text used for skill-overlap labels and feature construction.", "feature", "Safe after skill parsing and alias policy are versioned."),
    "requirements_concat": ("Concatenated requirement text used for requirement coverage and text embedding.", "feature", "Safe if source-only and not label-derived."),
    "language_signal": ("Detected job language used for language slices and output-language diagnostics.", "feature_quality", "Safe for audit/slicing; model use requires Phase 3 language policy."),
    "source_posted_at": ("Source posting timestamp used for freshness/staleness filtering.", "filter_only", "Use to exclude stale jobs; do not train as content feature."),
    "status": ("Job availability status used to filter inactive records.", "filter_only", "Required for candidate eligibility; do not train on inactive jobs."),
    "fit_input_quality_score": ("Derived data-quality score for whether job input is usable.", "quality_gate", "Use for filtering/audit only; derived quality fields can leak curation rules."),
    "fit_input_has_requirements": ("Derived flag showing requirement text availability.", "quality_gate", "Use for audit/filter only."),
    "fit_input_has_skills": ("Derived flag showing skill availability.", "quality_gate", "Use for audit/filter only."),
    "skills_count_total": ("Derived skill count used for feature-quality diagnostics.", "feature_quality", "Safe for diagnostics; model use requires missingness policy."),
    "description_length_chars": ("Description length used for feature-quality diagnostics.", "feature_quality", "Safe for diagnostics; avoid as direct model signal unless validated."),
    "req_len": ("Requirement text length used for feature-quality diagnostics.", "feature_quality", "Safe for diagnostics; avoid as direct model signal unless validated."),
    "_is_tech_cat": ("Internal tech-category flag from preprocessing.", "quality_gate", "Use for source filtering only; internal derived flags should not be target features."),
    "_is_tech_title": ("Internal tech-title flag from preprocessing.", "quality_gate", "Use for source filtering only."),
    "_is_tech_skills": ("Internal tech-skill flag from preprocessing.", "quality_gate", "Use for source filtering only."),
    "is_tech": ("Final tech-job eligibility flag.", "quality_gate", "Use for dataset scope filtering only."),
    "tech_signal_source": ("Explanation of why a job was considered tech-relevant.", "quality_gate", "Use for audit only."),
    "has_salary_info": ("Derived salary availability flag.", "feature_quality", "Use for audit/filtering only until salary policy exists."),
    "skills_clean": ("Cleaned skill list used for skill normalization and overlap features.", "feature", "Safe after alias policy and unknown-skill handling are versioned."),
    "is_train_ready": ("Training eligibility flag from preprocessing.", "quality_gate", "Use as inclusion gate only; not as model feature."),
}

PROFILE_FIELD_POLICIES = {
    "ID": ("Stable profile identifier used for grouping, joins, and leakage-safe splits.", "identifier_only", "Never use as model feature; use to isolate train/validation groups."),
    "Skills": ("Candidate skill list used for skill overlap, missing-skill analysis, and text construction.", "feature", "Safe after parsing, alias normalization, and unknown-skill policy."),
    "Projects": ("Project text used for semantic profile/CV evidence.", "feature", "Safe after text-quality and PII checks."),
    "Education": ("Education signal used for audit slices and optional feature experiments.", "use_with_caution", "Can encode bias; do not use in production scorer without fairness review."),
    "Experience": ("Candidate experience bucket used for experience-match labels and normalization checks.", "feature", "Safe only after Phase 3 maps every observed value."),
    "Job_Role": ("Candidate target/current role used for role-match evidence and slices.", "feature", "Safe after role taxonomy policy; avoid leaking desired output labels."),
    "Required_Skills": ("Role-required skill list attached to profile source data.", "use_with_caution", "Leakage-prone when it represents target role truth; use for audit/bootstrap labels only until validated."),
}

PAIR_FIELD_POLICIES = {
    "profile_id": ("Profile id in generated pair dataset.", "identifier_only", "Use for grouped split and leakage checks only."),
    "job_id": ("Job id in generated pair dataset.", "identifier_only", "Use for joins and backend candidate binding checks only."),
    "profile_text": ("Constructed profile text used for embedding/features.", "feature", "Safe only if construction excludes labels and wrapper-only fields."),
    "job_text": ("Constructed job text used for embedding/features.", "feature", "Safe only if source comes from active backend job fields."),
    "profile_skills": ("Profile skills used for skill-overlap label/bootstrap features.", "feature", "Safe after skill normalization policy."),
    "job_skills": ("Job skills used for skill-overlap label/bootstrap features.", "feature", "Safe after skill normalization policy."),
    "profile_exp": ("Profile experience bucket used for experience-match label/bootstrap features.", "feature", "Blocked until mapping handles current values."),
    "job_exp": ("Job experience level used for experience-match label/bootstrap features.", "feature", "Blocked until mapping handles current values."),
    "fit_score": ("Weak-label target from old rule-based pair generation.", "target_only", "Never use as input feature; prototype label only."),
}

MODEL_OWNED_FIELDS = [
    "jobFitAlignment.score",
    "jobFitAlignment.summarySignals",
    "jobFitAlignment.missingSignals",
    "jobFitAlignment.matchedSkills",
    "jobFitAlignment.missingSkills",
    "atsFriendliness.score",
    "atsFriendliness.detectedIssues",
    "overallImpression.score",
    "overallImpression.summary",
    "recommendations[].jobId",
    "recommendations[].matchScore",
    "recommendations[].matchLevel",
    "recommendations[].matchedSkills",
    "recommendations[].missingSkills",
    "recommendations[].rankingSignals",
]

WRAPPER_OR_BACKEND_FIELDS = [
    "topActionables",
    "sectionReviews",
    "generatedCv",
    "jobRecommendations[].title",
    "jobRecommendations[].companyName",
    "jobRecommendations[].reason",
    "jobRecommendations[].nextStep",
    "job detail hydration",
    "auth and ownership",
    "request validation",
    "persistence",
    "OpenAI wrapper orchestration",
]


def clean_field_name(name: str) -> str:
    return name.lstrip("\ufeff")


def read_csv_profile(path: Path) -> dict[str, Any]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        fields = [clean_field_name(field) for field in (reader.fieldnames or [])]
        rows = 0
        nulls = Counter()
        samples: dict[str, list[str]] = {field: [] for field in fields}
        values_for_type: dict[str, list[str]] = {field: [] for field in fields}
        distinct_values: dict[str, set[str]] = {field: set() for field in fields}
        for raw_row in reader:
            rows += 1
            row = {clean_field_name(key): value for key, value in raw_row.items()}
            for field in fields:
                value = (row.get(field) or "").strip()
                if value == "":
                    nulls[field] += 1
                    continue
                if len(samples[field]) < 3 and value not in samples[field]:
                    samples[field].append(value)
                if len(values_for_type[field]) < 1000:
                    values_for_type[field].append(value)
                if len(distinct_values[field]) <= 10000:
                    distinct_values[field].add(value)
        return {
            "path": str(path.relative_to(ROOT)),
            "rows": rows,
            "columns": fields,
            "nulls": dict(nulls),
            "samples": samples,
            "type_samples": values_for_type,
            "distinct_count_sample": {field: len(values) for field, values in distinct_values.items()},
        }


def infer_scalar_type(values: list[str]) -> str:
    if not values:
        return "unknown_empty"
    lowered = {value.lower() for value in values}
    if lowered <= {"true", "false", "0", "1"}:
        return "boolean_like"
    int_ok = True
    for value in values:
        try:
            int(value)
        except ValueError:
            int_ok = False
            break
    if int_ok:
        return "integer_like"
    float_ok = True
    for value in values:
        try:
            number = float(value)
            if not math.isfinite(number):
                float_ok = False
                break
        except ValueError:
            float_ok = False
            break
    if float_ok:
        return "number_like"
    if any("+00:00" in value or "T" in value or "-" in value[:10] for value in values):
        parsed = 0
        for value in values[:20]:
            candidate = value.replace("Z", "+00:00")
            try:
                datetime.fromisoformat(candidate)
                parsed += 1
            except ValueError:
                pass
        if parsed >= max(1, min(len(values[:20]), 3)):
            return "datetime_like"
    return "string"


def build_source_schema_review(dataset_name: str, profile: dict[str, Any], policies: dict[str, tuple[str, str, str]]) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for field in profile["columns"]:
        purpose, training_use, safety_note = policies.get(
            field,
            ("Unclassified source field; must be reviewed before use.", "blocked_until_reviewed", "No default training use allowed."),
        )
        null_count = int(profile["nulls"].get(field, 0))
        null_rate = round(null_count / profile["rows"], 6) if profile["rows"] else None
        rows.append({
            "dataset": dataset_name,
            "field": field,
            "inferred_type": infer_scalar_type(profile["type_samples"].get(field, [])),
            "null_count": null_count,
            "null_rate": null_rate,
            "null_policy": "required_current_snapshot" if null_count == 0 else "nullable_monitor_or_impute_before_training",
            "example_values": profile["samples"].get(field, []),
            "training_purpose": purpose,
            "training_use": training_use,
            "safe_to_use_during_training": training_use in {"feature", "feature_quality", "quality_gate", "filter_only", "target_only", "identifier_only"},
            "safety_note": safety_note,
        })
    return rows


def build_pair_schema_review() -> list[dict[str, Any]]:
    # Parquet dependencies are intentionally not required for Phase 1. The legacy pair schema
    # is documented from Phase 0/verification reports and the known pair generation artifact.
    rows = []
    for field, (purpose, training_use, safety_note) in PAIR_FIELD_POLICIES.items():
        rows.append({
            "dataset": "generated_pairs",
            "field": field,
            "inferred_type": "number_like" if field == "fit_score" else "string_or_text",
            "null_policy": "must_be_non_null_for_training_pairs",
            "example_values": [],
            "training_purpose": purpose,
            "training_use": training_use,
            "safe_to_use_during_training": training_use != "target_only",
            "safety_note": safety_note,
        })
    return rows


def load_openapi() -> dict[str, Any]:
    with OPENAPI_JSON.open() as handle:
        return json.load(handle)


def print_table(rows: list[dict[str, Any]], columns: list[str], max_rows: int | None = None) -> None:
    selected = rows if max_rows is None else rows[:max_rows]
    if not selected:
        print("(no rows)")
        return
    widths = {column: len(column) for column in columns}
    rendered: list[dict[str, str]] = []
    for row in selected:
        rendered_row = {}
        for column in columns:
            value = row.get(column, "")
            text = json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else str(value)
            if len(text) > 96:
                text = text[:93] + "..."
            rendered_row[column] = text
            widths[column] = max(widths[column], len(text))
        rendered.append(rendered_row)
    print(" | ".join(column.ljust(widths[column]) for column in columns))
    print(" | ".join("-" * widths[column] for column in columns))
    for row in rendered:
        print(" | ".join(row[column].ljust(widths[column]) for column in columns))
    if max_rows is not None and len(rows) > max_rows:
        print(f"... {len(rows) - max_rows} more rows")


## Step 1.1 — Source schema review

### Purpose
Document every source column, data type, null policy, example value, and whether it is safe to use during training.

### Required input
`legacy/dataset/indotech_job_cleaned.csv`, `legacy/dataset/techtalent_profile_cleaned.csv`, and the known generated-pair schema from `legacy/artifacts/pairs.parquet`/Phase 0.

### Action
Read source CSV headers and samples, infer lightweight data types, count nulls, attach training-purpose policies, and mark each field as feature, identifier-only, quality gate, filter-only, wrapper/backend-owned, target-only, caution, or blocked.

### Expected output
A schema review table covering every raw job field, raw profile field, and generated-pair training field with purpose, null policy, safety decision, and example values.

### Verification
All raw source columns must appear exactly once in the audit table. No unclassified raw field may be accepted for training.


In [8]:
job_profile = read_csv_profile(JOBS_CSV)
profile_profile = read_csv_profile(PROFILES_CSV)
source_schema_review = (
    build_source_schema_review("raw_jobs", job_profile, JOB_FIELD_POLICIES)
    + build_source_schema_review("raw_profiles", profile_profile, PROFILE_FIELD_POLICIES)
    + build_pair_schema_review()
)

unclassified_fields = [row for row in source_schema_review if row["training_use"] == "blocked_until_reviewed"]
field_review_counts = Counter(row["training_use"] for row in source_schema_review)

print(f"Raw job rows: {job_profile['rows']} | columns: {len(job_profile['columns'])}")
print(f"Raw profile rows: {profile_profile['rows']} | columns: {len(profile_profile['columns'])}")
print(f"Generated pair fields documented: {len(PAIR_FIELD_POLICIES)}")
print("Training-use distribution:")
print(json.dumps(dict(sorted(field_review_counts.items())), indent=2))
print("\nSchema review sample:")
print_table(
    source_schema_review,
    ["dataset", "field", "inferred_type", "null_policy", "training_use", "training_purpose", "safety_note"],
    max_rows=25,
)

assert len(unclassified_fields) == 0, f"Unclassified fields remain: {unclassified_fields}"
assert len([row for row in source_schema_review if row["dataset"] == "raw_jobs"]) == len(job_profile["columns"])
assert len([row for row in source_schema_review if row["dataset"] == "raw_profiles"]) == len(profile_profile["columns"])


Raw job rows: 2073 | columns: 35
Raw profile rows: 69929 | columns: 7
Generated pair fields documented: 9
Training-use distribution:
{
  "feature": 19,
  "feature_quality": 5,
  "filter_only": 2,
  "identifier_only": 4,
  "quality_gate": 9,
  "target_only": 1,
  "use_with_caution": 3,
  "wrapper_or_backend": 8
}

Schema review sample:
dataset  | field                      | inferred_type | null_policy               | training_use       | training_purpose                                                                                 | safety_note                                                                                    
-------- | -------------------------- | ------------- | ------------------------- | ------------------ | ------------------------------------------------------------------------------------------------ | -----------------------------------------------------------------------------------------------
raw_jobs | job_id                     | string        | require

## Step 1.2 — Identifier integrity check

### Purpose
Describe checks for profile IDs, job IDs, duplicates, stale records, inactive jobs, and leakage-prone joins.

### Required input
Raw job/profile CSVs, Phase 0 generated-pair row counts, and the backend candidate-boundary contract.

### Action
Count duplicate and missing source identifiers, summarize job status and training-readiness flags, review job freshness fields, and list leakage-prone joins or fields that must be excluded from model features.

### Expected output
Identifier integrity summary plus explicit leakage controls for pair generation, train/validation splits, recommendation output, and backend job hydration.

### Verification
Identifier checks must show whether source IDs are complete and unique. Leakage controls must block identifiers and weak-label targets from feature input.


In [9]:
def id_summary(profile: dict[str, Any], id_field: str) -> dict[str, Any]:
    seen: set[str] = set()
    duplicates = 0
    missing = 0
    with (ROOT / profile["path"]).open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        for raw_row in reader:
            row = {clean_field_name(key): value for key, value in raw_row.items()}
            value = (row.get(id_field) or "").strip()
            if not value:
                missing += 1
                continue
            if value in seen:
                duplicates += 1
            seen.add(value)
    return {
        "field": id_field,
        "rows": profile["rows"],
        "missing_ids": missing,
        "duplicate_ids": duplicates,
        "unique_ids": len(seen),
        "passed": missing == 0 and duplicates == 0,
    }


def value_distribution(path: Path, field: str) -> dict[str, int]:
    counts = Counter()
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        for raw_row in reader:
            row = {clean_field_name(key): value for key, value in raw_row.items()}
            value = (row.get(field) or "").strip() or "<missing>"
            counts[value] += 1
    return dict(counts.most_common())


def posted_at_summary(path: Path) -> dict[str, Any]:
    values: list[datetime] = []
    parse_errors = 0
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        for raw_row in reader:
            row = {clean_field_name(key): value for key, value in raw_row.items()}
            value = (row.get("source_posted_at") or "").strip()
            if not value:
                continue
            try:
                values.append(datetime.fromisoformat(value.replace("Z", "+00:00")))
            except ValueError:
                parse_errors += 1
    if not values:
        return {"count": 0, "parse_errors": parse_errors}
    latest = max(values)
    oldest = min(values)
    return {
        "count": len(values),
        "oldest": oldest.isoformat(),
        "latest": latest.isoformat(),
        "parse_errors": parse_errors,
        "freshness_policy": "Backend must exclude inactive or stale jobs before candidate retrieval; Phase 4 must document stale-record threshold before pair generation.",
    }

identifier_integrity = {
    "raw_jobs": id_summary(job_profile, "job_id"),
    "raw_profiles": id_summary(profile_profile, "ID"),
    "job_status_distribution": value_distribution(JOBS_CSV, "status"),
    "job_train_ready_distribution": value_distribution(JOBS_CSV, "is_train_ready"),
    "job_posted_at_summary": posted_at_summary(JOBS_CSV),
    "pair_artifact": {
        "path": str(PAIRS_PARQUET.relative_to(ROOT)),
        "schema_fields_documented": list(PAIR_FIELD_POLICIES),
        "phase0_rows": json.loads(PHASE0_REPORT.read_text())["dataset_snapshot"]["generated_pairs"]["rows"] if PHASE0_REPORT.exists() else None,
        "required_split_control": "Group validation by profile_id at minimum; consider job-family isolation before ranking experiments.",
    },
    "leakage_controls": [
        "Do not use job_id or profile_id as model features.",
        "Do not use fit_score as an input feature; it is the weak-label target only.",
        "Do not let profile Required_Skills leak target-role truth into validation labels without manual review.",
        "Filter inactive jobs before pair generation and before recommendation candidate scoring.",
        "Separate backend job detail hydration from model ranking output; model must not invent or return stale job details.",
        "Use grouped splits by profile_id and report duplicate profile/job pairs before baseline evaluation.",
    ],
}

print(json.dumps(identifier_integrity, indent=2, ensure_ascii=False))
assert identifier_integrity["raw_jobs"]["passed"]
assert identifier_integrity["raw_profiles"]["passed"]


{
  "raw_jobs": {
    "field": "job_id",
    "rows": 2073,
    "missing_ids": 0,
    "duplicate_ids": 0,
    "unique_ids": 2073,
    "passed": true
  },
  "raw_profiles": {
    "field": "ID",
    "rows": 69929,
    "missing_ids": 0,
    "duplicate_ids": 0,
    "unique_ids": 69929,
    "passed": true
  },
  "job_status_distribution": {
    "ACTIVE": 2072,
    "EXPIRED": 1
  },
  "job_train_ready_distribution": {
    "1": 2071,
    "0": 2
  },
  "job_posted_at_summary": {
    "count": 2073,
    "oldest": "2019-12-09T11:47:20.295000+00:00",
    "latest": "2026-05-18T00:35:36.607000+00:00",
    "parse_errors": 0,
    "freshness_policy": "Backend must exclude inactive or stale jobs before candidate retrieval; Phase 4 must document stale-record threshold before pair generation."
  },
  "pair_artifact": {
    "path": "legacy/artifacts/pairs.parquet",
    "schema_fields_documented": [
      "profile_id",
      "job_id",
      "profile_text",
      "job_text",
      "profile_skills",
      "job

## Step 1.3 — Output contract mapping

### Purpose
Map training outputs to product fields: `jobFitAlignment`, `atsFriendliness`, `overallImpression`, and recommendation scores.

### Required input
`references/docs/generated/openapi.json`, `references/docs/modules/ai-cv-analyzer.md`, `references/docs/modules/ai-job-fit.md`, `references/docs/modules/ai-job-recommendations.md`, and `references/docs/integrations/model-api.md`.

### Action
Map each product-facing field to the model/core signal or wrapper/backend owner, record score ranges, and identify whether current source data can support the field.

### Expected output
A contract mapping table that later phases can use as a stable output schema before labels, features, or models are changed.

### Verification
Every model-owned target in the Phase 1 TODO scope must be mapped separately from wrapper/backend-owned fields. Score fields must preserve the `0-100` API range.


In [10]:
openapi = load_openapi()
cv_schema = openapi["components"]["schemas"]["CvAnalysis"]["properties"]["analysisResult"]["properties"]

output_contract_mapping = [
    {
        "product_field": "analysisResult.jobFitAlignment.score",
        "api_shape": "integer 0-100",
        "owner": "model_core",
        "model_output": "jobFitAlignment.score",
        "training_signal_needed": "skill overlap, semantic similarity, experience match, role match, requirement coverage",
        "current_data_support": "partial_weak_label_only",
        "notes": "Current fit_score max is below high-fit range; needs Phase 2 label schema and Phase 4 balanced pairs.",
    },
    {
        "product_field": "analysisResult.jobFitAlignment.summary",
        "api_shape": "string",
        "owner": "wrapper_from_model_signals",
        "model_output": "summarySignals, matchedSkills, missingSkills, confidence notes",
        "training_signal_needed": "grounded matched and missing signals from source text/skills",
        "current_data_support": "partial_skill_signals_only",
        "notes": "Wrapper may render final copy, but model/core must expose grounded signals.",
    },
    {
        "product_field": "analysisResult.atsFriendliness.score",
        "api_shape": "integer 0-100",
        "owner": "model_core_or_rule_baseline",
        "model_output": "atsFriendliness.score",
        "training_signal_needed": "CV parseability, section completeness, contact/date detection, metric evidence, formatting risk",
        "current_data_support": "blocked_no_cv_benchmark",
        "notes": "No controlled CV benchmark exists in current data snapshot.",
    },
    {
        "product_field": "analysisResult.atsFriendliness.summary",
        "api_shape": "string",
        "owner": "wrapper_from_model_signals",
        "model_output": "detectedIssues",
        "training_signal_needed": "issue taxonomy and evaluated issue detector",
        "current_data_support": "blocked_no_ats_labels",
        "notes": "Issue taxonomy required before classifier or scored rule set.",
    },
    {
        "product_field": "analysisResult.overallImpression",
        "api_shape": "string",
        "owner": "wrapper_from_model_signals",
        "model_output": "overallImpression.score, overallImpression.summary, confidence notes",
        "training_signal_needed": "grounded alignment strengths, gaps, ATS risks, fallback cases",
        "current_data_support": "partial_rule_template_only",
        "notes": "Must not hallucinate unsupported skills or seniority.",
    },
    {
        "product_field": "analysisResult.jobRecommendations[].matchScore",
        "api_shape": "integer 0-100, max 5 items in CV Analyzer response",
        "owner": "model_core_for_candidate_ranking",
        "model_output": "recommendations[].matchScore for backend-provided jobCandidates",
        "training_signal_needed": "candidate job set, ranking labels/baselines, no unknown job IDs",
        "current_data_support": "partial_static_job_index_only",
        "notes": "Backend owns candidate retrieval and final job detail hydration.",
    },
]

openapi_score_constraints = {
    "jobFitAlignment.score": cv_schema["jobFitAlignment"]["properties"]["score"],
    "atsFriendliness.score": cv_schema["atsFriendliness"]["properties"]["score"],
    "jobRecommendations[].matchScore": cv_schema["jobRecommendations"]["items"]["properties"]["matchScore"],
}

print_table(
    output_contract_mapping,
    ["product_field", "api_shape", "owner", "model_output", "current_data_support", "notes"],
)
print("\nOpenAPI score constraints:")
print(json.dumps(openapi_score_constraints, indent=2))

for key, constraint in openapi_score_constraints.items():
    assert constraint.get("minimum") == 0 and constraint.get("maximum") == 100, key
assert {"jobFitAlignment", "atsFriendliness", "overallImpression"} <= {field.split(".")[1] for field in [row["product_field"] for row in output_contract_mapping] if field.startswith("analysisResult.")}


product_field                                  | api_shape                                          | owner                            | model_output                                                         | current_data_support          | notes                                                                                           
---------------------------------------------- | -------------------------------------------------- | -------------------------------- | -------------------------------------------------------------------- | ----------------------------- | ------------------------------------------------------------------------------------------------
analysisResult.jobFitAlignment.score           | integer 0-100                                      | model_core                       | jobFitAlignment.score                                                | partial_weak_label_only       | Current fit_score max is below high-fit range; needs Phase 2 label schema and Phase 4 

## Step 1.4 — Boundary definition

### Purpose
Separate model-owned outputs from wrapper-owned outputs such as `topActionables`, `sectionReviews`, and hydrated job details.

### Required input
Phase 1 scope, Model API integration documentation, CV Analyzer documentation, and OpenAPI response fields.

### Action
Create a durable owner matrix for model/core, API wrapper, and backend responsibilities. Mark fields that must stay out of training labels or model artifacts.

### Expected output
Boundary table and out-of-scope field list that prevent future notebooks from training toward wrapper/backend responsibilities.

### Verification
The boundary table must explicitly classify all Phase 1 TODO outputs and wrapper-owned fields.


In [11]:
boundary_definition = [
    {
        "area": "CV analysis core",
        "field_or_behavior": "jobFitAlignment.score and grounded matched/missing alignment signals",
        "owner": "model_core",
        "training_action": "define labels/features in Phases 2-6",
        "must_not_do": "Do not produce final public copy without wrapper validation.",
    },
    {
        "area": "CV quality",
        "field_or_behavior": "atsFriendliness.score and detectedIssues",
        "owner": "model_core_or_rule_baseline",
        "training_action": "define ATS labels and benchmark in Phases 2 and 7",
        "must_not_do": "Do not claim ATS production quality without CV benchmark evidence.",
    },
    {
        "area": "Summary signal",
        "field_or_behavior": "overallImpression grounded summary signal",
        "owner": "model_core_signal + wrapper_rendering",
        "training_action": "define grounded templates and fallback policy in Phase 8",
        "must_not_do": "Do not infer skills, seniority, language, or impact not present in evidence.",
    },
    {
        "area": "Actionable copy",
        "field_or_behavior": "topActionables",
        "owner": "api_wrapper",
        "training_action": "consume model signals only",
        "must_not_do": "Do not train core model to own product copy or prompt orchestration.",
    },
    {
        "area": "Section-level review",
        "field_or_behavior": "sectionReviews",
        "owner": "api_wrapper",
        "training_action": "consume detected sections/issues if available",
        "must_not_do": "Do not force missing CV sections into response.",
    },
    {
        "area": "Job recommendations",
        "field_or_behavior": "recommendation score/rank for backend-provided candidates",
        "owner": "model_core_ranker",
        "training_action": "define candidate ranking contract in Phase 9",
        "must_not_do": "Do not rank static artifact jobs as final public recommendations.",
    },
    {
        "area": "Job detail hydration",
        "field_or_behavior": "job title, companyName, location, bookmark/application status, visibility",
        "owner": "backend_api",
        "training_action": "use jobId only for binding/validation",
        "must_not_do": "Do not embed stale job details in model artifact output.",
    },
    {
        "area": "Security and persistence",
        "field_or_behavior": "auth, ownership, storage, raw CV retention, request validation",
        "owner": "backend_api",
        "training_action": "respect privacy constraints and avoid raw CV logs",
        "must_not_do": "Do not train on tokens, auth data, storage keys, or unrelated PII.",
    },
]

out_of_scope_for_model_training = WRAPPER_OR_BACKEND_FIELDS

print_table(boundary_definition, ["area", "field_or_behavior", "owner", "training_action", "must_not_do"])
print("\nOut of scope for model training:")
for item in out_of_scope_for_model_training:
    print(f"- {item}")

owners = {row["owner"] for row in boundary_definition}
assert "model_core" in owners or "model_core_ranker" in owners
assert "api_wrapper" in owners
assert "backend_api" in owners


area                     | field_or_behavior                                                         | owner                                 | training_action                                          | must_not_do                                                                 
------------------------ | ------------------------------------------------------------------------- | ------------------------------------- | -------------------------------------------------------- | ----------------------------------------------------------------------------
CV analysis core         | jobFitAlignment.score and grounded matched/missing alignment signals      | model_core                            | define labels/features in Phases 2-6                     | Do not produce final public copy without wrapper validation.                
CV quality               | atsFriendliness.score and detectedIssues                                  | model_core_or_rule_baseline           | define ATS labels an

## Step 1.5 — Data readiness decision

### Purpose
State whether the available data can support the next phase or whether synthetic/manual labels are needed first.

### Required input
Schema review, identifier integrity checks, output contract mapping, boundary definition, and Phase 0 baseline risks.

### Action
Evaluate readiness separately for Phase 2 label-schema design, model training, ATS scoring, overall-impression generation, and recommendation ranking. Write the full Phase 1 report artifact.

### Expected output
Explicit readiness decision with blockers, required labels, and acceptance-status flags.

### Verification
The decision must be explicit and must not mark production training ready while labels, high-fit coverage, ATS benchmark, and backend candidate flow remain incomplete.


In [12]:
data_readiness_decision = {
    "ready_for_phase_2_label_schema_design": True,
    "ready_for_new_model_training": False,
    "ready_for_production_readiness_claim": False,
    "manual_or_synthetic_labels_needed_first": True,
    "decision": "Proceed to Phase 2 label schema and baseline definitions; do not start new model training yet.",
    "evidence": [
        "Raw job and profile identifiers are complete and unique in the current snapshot.",
        "Every raw training input field has a documented purpose and safety decision.",
        "Current generated fit_score is weak-label only and lacks high-fit coverage from Phase 0.",
        "ATS friendliness lacks controlled CV benchmark samples and issue labels.",
        "Recommendation flow must switch from static job artifacts to backend-provided candidates before production ranking evaluation.",
        "Wrapper-owned fields are separated from model/core outputs.",
    ],
    "required_before_training": [
        "Phase 2 label schema for job-fit, ATS, score bands, and manual validation samples.",
        "Phase 3 normalization rules for experience, skills, language, and text construction.",
        "Phase 4 balanced pair generation with leakage-safe splits and high/medium/low coverage.",
        "Phase 5 baseline evaluation before complex model experiments.",
        "ATS benchmark and labels before claiming atsFriendliness production support.",
        "Candidate job input contract before recommendation ranking model evaluation.",
    ],
}

acceptance = {
    "every_training_input_field_has_documented_purpose": len(unclassified_fields) == 0,
    "model_owned_outputs_separated_from_api_wrapper_outputs": bool(boundary_definition and out_of_scope_for_model_training),
    "data_readiness_decision_is_explicit": bool(data_readiness_decision["decision"]),
}

phase1_report = {
    "schema_version": "phase-01-data-audit-contracts-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_snapshot": "legacy",
    "inputs": {
        "jobs_csv": str(JOBS_CSV.relative_to(ROOT)),
        "profiles_csv": str(PROFILES_CSV.relative_to(ROOT)),
        "pairs_parquet": str(PAIRS_PARQUET.relative_to(ROOT)),
        "openapi_json": str(OPENAPI_JSON.relative_to(ROOT)),
        "phase0_report": str(PHASE0_REPORT.relative_to(ROOT)),
    },
    "source_schema_review": source_schema_review,
    "identifier_integrity": identifier_integrity,
    "output_contract_mapping": output_contract_mapping,
    "openapi_score_constraints": openapi_score_constraints,
    "boundary_definition": boundary_definition,
    "model_owned_fields": MODEL_OWNED_FIELDS,
    "wrapper_or_backend_fields": WRAPPER_OR_BACKEND_FIELDS,
    "data_readiness_decision": data_readiness_decision,
    "acceptance": acceptance,
}

REPORTS.mkdir(parents=True, exist_ok=True)
PHASE1_REPORT.write_text(json.dumps(phase1_report, indent=2, ensure_ascii=False) + "\n")

print(json.dumps(data_readiness_decision, indent=2, ensure_ascii=False))
print("\nAcceptance:")
print(json.dumps(acceptance, indent=2))
print(f"Wrote {PHASE1_REPORT.relative_to(ROOT)}")
assert all(acceptance.values())
assert not data_readiness_decision["ready_for_new_model_training"]


{
  "ready_for_phase_2_label_schema_design": true,
  "ready_for_new_model_training": false,
  "ready_for_production_readiness_claim": false,
  "manual_or_synthetic_labels_needed_first": true,
  "decision": "Proceed to Phase 2 label schema and baseline definitions; do not start new model training yet.",
  "evidence": [
    "Raw job and profile identifiers are complete and unique in the current snapshot.",
    "Every raw training input field has a documented purpose and safety decision.",
    "Current generated fit_score is weak-label only and lacks high-fit coverage from Phase 0.",
    "ATS friendliness lacks controlled CV benchmark samples and issue labels.",
    "Recommendation flow must switch from static job artifacts to backend-provided candidates before production ranking evaluation.",
    "Wrapper-owned fields are separated from model/core outputs."
  ],
  "required_before_training": [
    "Phase 2 label schema for job-fit, ATS, score bands, and manual validation samples.",
    "

## Acceptance criteria

- [x] Every training input field has a documented purpose.
- [x] Model-owned outputs are separated from API-wrapper outputs.
- [x] Data readiness decision is explicit.

## Phase notes

- Phase 1 is ready for Phase 2 label-schema work, not model training.
- Current job/profile identifiers are complete and unique, but pair generation still needs grouped split and duplicate-pair checks in Phase 4.
- Profile `Required_Skills` is leakage-prone and must not become validation truth without manual review.
- Current `fit_score` remains a weak-label target only.
- `atsFriendliness` is blocked by missing CV benchmark data and issue labels.
- Recommendation ranking must use backend-provided candidate jobs; static `job_index.json` cannot own final public job recommendations.
